In [10]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def preprocess_data(data):
    """Creating new features for higher accuracy"""
    #Create new features
    data['Title'] = data['Name'].str.extract(' ([A-Za-z]+)\.').fillna('')
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    data['FarePerPerson'] = data['Fare'] / data['FamilySize'].replace(0, 1)
    data['IsAlone'] = (data['FamilySize'] == 1).astype(int)
    data['HasCabin'] = data['Cabin'].notnull().astype(int)
    data['Deck'] = data['Cabin'].str[0].fillna('U')

    #simplify titles, grouping rare together
    titles = ['Mr', 'Mrs', 'Miss', 'Master']
    data.loc[~data['Title'].isin(titles), 'Title'] = 'Rare'

    #Bin age to categories
    age_groups = [0, 13, 19, 60, 120]
    labels = ['Child', 'Teen', 'Adult', 'Senior']
    data['AgeGroup'] = pd.cut(data['Age'], bins = age_groups, 
                              labels = labels, right = False)
    return data

def build_pipeline(numerical_features, int_features, categorical_features):
    """Builds preprocessing pipeline for numeric, integer and categorical data."""
    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy = 'median')),
        ('scaler', StandardScaler())
    ])
    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy = 'most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown = 'ignore', sparse_output = False))
    ])
    int_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy = 'median'))
    ])
    full_pipeline = ColumnTransformer([
        ('num', num_pipeline, numerical_features),
        ('cat', cat_pipeline, categorical_features),
        ('int', int_pipeline, int_features)
    ])
    return full_pipeline
    
def train_xgboost(train_x, train_y):
    """Train XGBoost using GridSearchCv with hyperparameters"""
    #find parameters to get highest accuracy
    param_grid = {
        'n_estimators': [200],
        'max_depth': [3, 4],
        'learning_rate': [0.03, 0.05],
        'subsample': [0.8, 1],
        'colsample_bytree': [0.6, 0.8],
        'reg_alpha': [0, 0.3],
        'reg_lambda': [1],
        'min_child_weight': [1, 3],
    }
    model = xgb.XGBClassifier(use_label_encoder = False,
                              eval_metric = 'logloss',
                              random_state = 1)
    grid_search = GridSearchCV(model, param_grid, cv = 5, scoring = 'accuracy', n_jobs = -1)
    grid_search.fit(train_x, train_y)
    return grid_search

def evaluate_predictions(model, x, y):
    """Evaluate model predictions with accuracy score"""
    predictions = model.predict(x)
    return accuracy_score(y, predictions)

def report_model_performance(grid_search, train_acc, val_acc):
    """Print model performance summary"""
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Training model accuracy: {train_acc:.4f}")
    print(f"Cross-validation accuracy: {grid_search.best_score_:.4f}")
    print(f"Validation accuracy: {val_acc:.4f}")

def main():
    #load data files
    train_data = pd.read_csv('/kaggle/input/titanic/train.csv')
    test_data = pd.read_csv('/kaggle/input/titanic/test.csv')

    #combine train and test dataset in one to improve fitting
    combined_data = pd.concat([train_data.drop('Survived', axis = 1),
                                      test_data], ignore_index = True)

    #Apply feature engineering and preprocessing
    processed_data = preprocess_data(combined_data)

    #define features to use
    numerical_features = []
    categorical_features = ['Sex', 'Pclass', 'Title', 'AgeGroup']
    int_features = ['FamilySize']
    
    #select features used for training
    selected_features = numerical_features + categorical_features + int_features

    train_size = len(train_data)
    
    #split data into train and test sets
    X = processed_data.iloc[:train_size][selected_features]
    y = train_data['Survived']
    X_test_final = processed_data.iloc[train_size:][selected_features]

    preprocessor = build_pipeline(numerical_features, int_features, categorical_features)

    X_processed = preprocessor.fit_transform(X)
    
    train_x, test_x, train_y, test_y = train_test_split(X_processed, y, stratify = y, test_size = 0.2,
                                                        random_state = 1)
    #train XGBoost algorithm
    grid_search = train_xgboost(train_x, train_y)

    #find most optimal features from model
    best_model = grid_search.best_estimator_
    feature_importances = best_model.feature_importances_

    #evaluate model on train and validation set
    train_acc = evaluate_predictions(best_model, train_x, train_y)
    val_acc = evaluate_predictions(best_model, test_x, test_y)

    #show model performance data
    report_model_performance(grid_search, train_acc, val_acc)

    #make predictions on the test set
    X_test_prepared = preprocessor.transform(X_test_final)
    predictions = best_model.predict(X_test_prepared)

    #save submission file
    submission = pd.DataFrame({'PassengerId': test_data['PassengerId'], 
                              'Survived': predictions})
    submission.to_csv('submission.csv', index = False)

if __name__ == "__main__":
    main()

Best parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 200, 'reg_alpha': 0.3, 'reg_lambda': 1, 'subsample': 1}
Training model accuracy: 0.8315
Cross-validation accuracy: 0.8244
Validation accuracy: 0.8492
